<a href="https://colab.research.google.com/github/prasertrak/Advanced-Data-Engineering-and-Applied-Analytics/blob/main/Part3_data_cleaning_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Part 3 — Data Cleaning & Validation**
## Order Fulfillment Analytics Pipeline


### Learning Objectives
- Detect dirty data
- Clean invalid records
- Handle NULL values
- Standardize text
- Quarantine bad records
- Perform reconciliation


In [ ]:
import pandas as pd

**Example 1 - Create Mock Dirty Dataset**

In [ ]:
data = {
    "order_id": [
        "ORD001","ORD002","ORD003","ORD004","ORD005"
    ],
    "order_date": [
        "2026-05-01",
        "03/05/2026",
        "2026-13-01",
        "2026/05/04",
        "2026-05-05"
    ],
    "status": [
        "D",
        None,
        "X",
        "S",
        "P"
    ],
    "total_amount": [
        1200,
        -500,
        1500,
        900,
        2000
    ],
    "province": [
        "bangkok",
        "Chiangmai",
        "PHUKET",
        "chonburi",
        "Bangkok"
    ]
}

df = pd.DataFrame(data)

df

,order_id,order_date,status,total_amount,province
0,ORD001,2026-05-01,D,1200,bangkok
1,ORD002,03/05/2026,None,-500,Chiangmai
2,ORD003,2026-13-01,X,1500,PHUKET
3,ORD004,2026/05/04,S,900,chonburi
4,ORD005,2026-05-05,P,2000,Bangkok


In [ ]:
df.to_csv(
    "dirty_orders.csv",
    index=False
)

In [ ]:
!ls

dirty_orders.csv  sample_data


**Example 2 - Fill Null Status**

In [ ]:
df["status"] = df["status"].fillna("UNKNOWN")

df

,order_id,order_date,status,total_amount,province
0,ORD001,2026-05-01,D,1200,bangkok
1,ORD002,03/05/2026,UNKNOWN,-500,Chiangmai
2,ORD003,2026-13-01,X,1500,PHUKET
3,ORD004,2026/05/04,S,900,chonburi
4,ORD005,2026-05-05,P,2000,Bangkok


**Example 3 - Replace Invalid Status Codes**

In [ ]:
valid_status = ["P", "S", "D", "C"]

#df.loc[row_indexer, column_indexer]
df.loc[
    ~df["status"].isin(valid_status),
    "status"
] = "UNKNOWN"

df

,order_id,order_date,status,total_amount,province
0,ORD001,2026-05-01,D,1200,bangkok
1,ORD002,03/05/2026,UNKNOWN,-500,Chiangmai
2,ORD003,2026-13-01,UNKNOWN,1500,PHUKET
3,ORD004,2026/05/04,S,900,chonburi
4,ORD005,2026-05-05,P,2000,Bangkok


**Example 4 - Standardize Province Names**

In [ ]:
df["province"] = df["province"].str.upper()

df

,order_id,order_date,status,total_amount,province
0,ORD001,2026-05-01,D,1200,BANGKOK
1,ORD002,03/05/2026,UNKNOWN,-500,CHIANGMAI
2,ORD003,2026-13-01,UNKNOWN,1500,PHUKET
3,ORD004,2026/05/04,S,900,CHONBURI
4,ORD005,2026-05-05,P,2000,BANGKOK


**Example 5 - Convert Date Format**


In [ ]:
df["order_date"] = pd.to_datetime(
    df["order_date"],
    errors="coerce"
)

df

,order_id,order_date,status,total_amount,province
0,ORD001,2026-05-01,D,1200,BANGKOK
1,ORD002,NaT,UNKNOWN,-500,CHIANGMAI
2,ORD003,NaT,UNKNOWN,1500,PHUKET
3,ORD004,NaT,S,900,CHONBURI
4,ORD005,2026-05-05,P,2000,BANGKOK


**Example 6 - Detect Invalid Dates**

In [ ]:
invalid_dates = df[
    df["order_date"].isnull()
]

invalid_dates

,order_id,order_date,status,total_amount,province
1,ORD002,NaT,UNKNOWN,-500,CHIANGMAI
2,ORD003,NaT,UNKNOWN,1500,PHUKET
3,ORD004,NaT,S,900,CHONBURI


**Example 7 - Detect Negative Amount**


In [ ]:
df

,order_id,order_date,status,total_amount,province
0,ORD001,2026-05-01,D,1200,BANGKOK
1,ORD002,NaT,UNKNOWN,-500,CHIANGMAI
2,ORD003,NaT,UNKNOWN,1500,PHUKET
3,ORD004,NaT,S,900,CHONBURI
4,ORD005,2026-05-05,P,2000,BANGKOK


In [ ]:
invalid_amount = df[
    df["total_amount"] < 0
]

invalid_amount

,order_id,order_date,status,total_amount,province
1,ORD002,NaT,UNKNOWN,-500,CHIANGMAI


**Example 8 - Create Validation Flag**


In [ ]:
df

,order_id,order_date,status,total_amount,province
0,ORD001,2026-05-01,D,1200,BANGKOK
1,ORD002,NaT,UNKNOWN,-500,CHIANGMAI
2,ORD003,NaT,UNKNOWN,1500,PHUKET
3,ORD004,NaT,S,900,CHONBURI
4,ORD005,2026-05-05,P,2000,BANGKOK


In [ ]:
df["is_valid"] = True

In [ ]:
df

,order_id,order_date,status,total_amount,province,is_valid
0,ORD001,2026-05-01,D,1200,BANGKOK,True
1,ORD002,NaT,UNKNOWN,-500,CHIANGMAI,True
2,ORD003,NaT,UNKNOWN,1500,PHUKET,True
3,ORD004,NaT,S,900,CHONBURI,True
4,ORD005,2026-05-05,P,2000,BANGKOK,True


In [ ]:

df.loc[
    df["order_date"].isnull(),
    "is_valid"
] = False

df.loc[
    df["total_amount"] < 0,
    "is_valid"
] = False

df

,order_id,order_date,status,total_amount,province,is_valid
0,ORD001,2026-05-01,D,1200,BANGKOK,True
1,ORD002,NaT,UNKNOWN,-500,CHIANGMAI,False
2,ORD003,NaT,UNKNOWN,1500,PHUKET,False
3,ORD004,NaT,S,900,CHONBURI,False
4,ORD005,2026-05-05,P,2000,BANGKOK,True


**Example 9 - Separate Clean Data and Quarantine Data**


In [ ]:
clean_df = df[
    df["is_valid"] == True
]

quarantine_df = df[
    df["is_valid"] == False
]

print("CLEAN DATA")
display(clean_df)

print("QUARANTINE DATA")
display(quarantine_df)


CLEAN DATA


,order_id,order_date,status,total_amount,province,is_valid
0,ORD001,2026-05-01,D,1200,BANGKOK,True
4,ORD005,2026-05-05,P,2000,BANGKOK,True


QUARANTINE DATA


,order_id,order_date,status,total_amount,province,is_valid
1,ORD002,NaT,UNKNOWN,-500,CHIANGMAI,False
2,ORD003,NaT,UNKNOWN,1500,PHUKET,False
3,ORD004,NaT,S,900,CHONBURI,False


**Example 10 - Row Count Reconciliation**


In [ ]:
before_count = len(df)
after_count = len(clean_df)

print("Before Cleaning:", before_count)
print("After Cleaning:", after_count)
print("Rejected Rows:", before_count - after_count)


Before Cleaning: 5
After Cleaning: 2
Rejected Rows: 3


**Example 11 - Revenue Reconciliation**


In [ ]:
before_total = df["total_amount"].sum()

after_total_clean = clean_df["total_amount"].sum()

after_total_rejected = quarantine_df["total_amount"].sum()

print("Before Cleaning Revenue:", before_total)
print("After Cleaning Revenue:", after_total_clean)
print("Rejected Revenue:", after_total_rejected)


Before Cleaning Revenue: 5100
After Cleaning Revenue: 3200
Rejected Revenue: 1900


**Example 12 - Export Clean Dataset**


In [ ]:
clean_df.to_csv(
    "clean_orders.csv",
    index=False
)

quarantine_df.to_csv(
    "quarantine_orders.csv",
    index=False
)

print("Export Completed")


Export Completed


**Example 13 - Full Cleaning Pipeline**

End-to-End Cleaning Workflow

In [ ]:
import pandas as pd

# load source data
df = pd.read_csv("dirty_orders.csv")

# fill null status
df["status"] = df["status"].fillna(
    "UNKNOWN"
)

# standardize province
df["province"] = (
    df["province"]
    .str.upper()
)

# convert dates
df["order_date"] = pd.to_datetime(
    df["order_date"],
    errors="coerce"
)

# create validation flag
df["is_valid"] = True

# invalid amount
df.loc[
    df["total_amount"] < 0,
    "is_valid"
] = False

# invalid dates
df.loc[
    df["order_date"].isnull(),
    "is_valid"
] = False

# quarantine records
quarantine_df = df[
    df["is_valid"] == False
]

# clean records
clean_df = df[
    df["is_valid"] == True
]

# export outputs
clean_df.to_csv(
    "clean_orders.csv",
    index=False
)

quarantine_df.to_csv(
    "quarantine_orders.csv",
    index=False
)

print("Cleaning Pipeline Completed")

Cleaning Pipeline Completed


In [ ]:
!ls

clean_orders.csv  dirty_orders.csv  quarantine_orders.csv  sample_data



# Final Learning Outcome

ผู้เรียนควรเข้าใจ:
- Data Cleaning
- Validation
- Quarantine
- Reconciliation
- Data Quality Thinking
